# pyceles Notebook: CELES_MAIN Replication + Beyond CELES

This notebook has two parts:

1. **CELES-like workflow** (minimal API surface)
   - Gaussian beam, normal incidence
   - GMRES solver
   - CELES-style mixed precision (`compute_dtype='complex64'`, `accum_dtype='complex128'`)
   - near-field, Poynting, far-field, and HDF5 save

2. **Beyond CELES features**
   - tilted Jones-polarized beam
   - multi-source solve (`Simulation.solve_sources(...) + Simulation.postprocess_sources(...)`)
   - channel-aware near/far-field analysis
   - unpolarized diagnostics and rich HDF5 outputs


In [ ]:
import sys
from pathlib import Path

# Make the notebook runnable from a source checkout without editable install.
def _find_repo_root(start: Path) -> Path:
    for cand in (start, *start.parents):
        if (cand / 'pyproject.toml').exists() and (cand / 'src' / 'pyceles').exists() and (cand / 'examples' / 'sphere_parameters.txt').exists():
            return cand
    raise FileNotFoundError('Could not find repository root containing src/pyceles and examples/sphere_parameters.txt.')

_repo_root = _find_repo_root(Path.cwd().resolve())
_src_path = _repo_root / 'src'
if str(_src_path) not in sys.path:
    sys.path.insert(0, str(_src_path))

import numpy as np
import matplotlib.pyplot as plt
import pyceles as pcl


## 1) CELES-like Minimal Workflow

In [ ]:
# --- Geometry + CELES-like simulation ingredients ---

sphere_data = np.loadtxt(_repo_root / 'examples' / 'sphere_parameters.txt')
n_particles = 500
radii = sphere_data[:n_particles, 0].astype(float)
positions = sphere_data[:n_particles, 1:4].astype(float)
n_particle = (sphere_data[:n_particles, 4] + 1j * sphere_data[:n_particles, 5]).astype(np.complex128)

source = pcl.GaussianBeam(
    wavelength=550.0,
    medium_n=1.0 + 0j,
    polarization='TE',
    polar_angle=0.0,
    azimuthal_angle=0.0,
    beam_width=2000.0,
    focal_point=(0.0, 0.0, 0.0),
    amplitude=1.0,
)

cfg = pcl.SimulationConfig(
    wavelength=550.0,
    n_medium=1.0 + 0j,
    lmax=3,
    source=source,
    polar_angles=pcl.core.uniform_polar_grid(721),
    azimuthal_angles=pcl.core.uniform_periodic_azimuth_grid(360),
    solver_method='gmres',
    solver_rtol=5e-4,
    solver_preconditioner_kind='grid_block',
    solver_preconditioner_subdivisions=(2, 2, 2),
    compute_dtype='complex64',
    accum_dtype='complex128',
    verbose=True,
)

sim = pcl.Simulation(cfg, positions=positions, radii=radii, n_particle=n_particle)


In [ ]:
# --- Run simulation (solve + far field + power diagnostics) ---

run = sim.run()
res = run.solver_result

print('Solver method:', res.method)
print('Solver info:', res.info)
print('Solver iterations:', res.iterations)
print('Relative residual:', res.relative_residual)
print('||x||:', np.linalg.norm(res.x))

if run.power is not None:
    P0 = run.power['P_initial']
    Pf = run.power['P_transmitted']
    Pb = run.power['P_reflected']
    rem = 1.0 - (Pf + Pb) / P0

    print(f'initial power:     {P0:.6e}')
    print(f'T (transmitted):   {Pf / P0 * 100:.4f} %')
    print(f'R (reflected):     {Pb / P0 * 100:.4f} %')
    print(f'1 - T + R closure: {rem * 100:.4f} %')


In [ ]:
# --- Near-field cross-cut + 8-panel map + Poynting ---

nf_slice = pcl.compute_near_field_slice(
    run,
    x_min=-4000.0,
    x_max=4000.0,
    z_min=-3000.0,
    z_max=5000.0,
    dx=40.0,
    plane='y',
    plane_value=0.0,
    channel='mixed',
    show_progress=True,
)

X = nf_slice.axis_0
Z = nf_slice.axis_1
E, H = nf_slice.field_maps['total']

fig, _ = pcl.io.plot_nearfield_panels(
    X,
    Z,
    E,
    H,
    run.positions,
    run.radii,
    plane=nf_slice.plane,
    plane_value=nf_slice.plane_value,
    real_limits=(-2.0, 2.0),
    abs_limits=(0.0, 2.0),
)
plt.show()

fig, _ = pcl.io.plot_nearfield_poynting_overlay(
    X,
    Z,
    E,
    H,
    run.positions,
    run.radii,
    plane=nf_slice.plane,
    plane_value=nf_slice.plane_value,
    stride=5,
    quiver_color='w',
)
plt.show()


In [ ]:
# --- Far-field hemispheres ---

ff = run.farfield
if ff.total_te is not None and ff.total_tm is not None:
    pwp_te, pwp_tm = ff.total_te, ff.total_tm
else:
    pwp_te, pwp_tm = ff.scattered_te, ff.scattered_tm

Iff = pcl.io.far_field_intensity(pwp_te, pwp_tm)
fig, _ = pcl.io.plot_farfield_hemispheres(
    polar_angles=cfg.polar_angles,
    azimuthal_angles=cfg.azimuthal_angles,
    intensity=Iff,
    cmap='inferno',
    independent_scales=True,
)
plt.show()


## 2) Beyond CELES

This section showcases features beyond original CELES:
- Jones polarization input (`(a_te, a_tm)`) and tilted beam incidence.
- Multi-source solve (`Simulation.solve_sources(...) + Simulation.postprocess_sources(...)`) for TE/TM channel outputs.
- Dense direct solver (`solver_method='direct'`) for small/moderate unknown counts.
  For large systems, matrix-free iterative solvers remain the scalable option.
- Precision policy (`compute_dtype` / `accum_dtype`) configurable independently.

For near fields below, we evaluate TE/TM basis slices once and then:
- coherently mix them into the requested Jones channel,
- compute unpolarized maps as an incoherent intensity average
  `0.5 * (|E_TE|^2 + |E_TM|^2)`.

Jones amplitudes are user-controlled; pyceles does not auto-normalize them.
In this notebook we use unit-power weights `(1, 1j)/sqrt(2)`.


In [ ]:
# --- Tilted Jones-polarized source and basis solve ---

source_ext = pcl.GaussianBeam(
    wavelength=550.0,
    medium_n=1.0 + 0j,
    polarization=((1.0 + 0j) / np.sqrt(2.0), 1.0j / np.sqrt(2.0)),  # normalized circular-like Jones mix (TE, TM)
    polar_angle=np.pi / 4.0,
    azimuthal_angle=np.pi / 7.0,
    beam_width=2000.0,
    focal_point=(0.0, 0.0, 0.0),
    amplitude=1.0,
)

cfg_ext = pcl.SimulationConfig(
    wavelength=550.0,
    n_medium=1.0 + 0j,
    lmax=3,
    source=source_ext,
    polar_angles=pcl.core.uniform_polar_grid(721),
    azimuthal_angles=pcl.core.uniform_periodic_azimuth_grid(360),
    solver_method='direct',
    compute_dtype='complex64',
    accum_dtype='complex128',
    solve_polarization_basis=True,  # convenience wrapper over solve_sources(...)
    verbose=True,
)

sim_ext = pcl.Simulation(cfg_ext, positions=positions, radii=radii, n_particle=n_particle)
run_ext = sim_ext.run()

print('Jones coefficients (a_te, a_tm):', run_ext.polarization_jones)
print('Basis available:', run_ext.coeffs_basis is not None)
print('Unpolarized diagnostics keys:', None if run_ext.unpolarized is None else list(run_ext.unpolarized.keys()))


In [ ]:
# --- Near field: requested Jones channel + basis + unpolarized intensity ---

nf_te = pcl.compute_near_field_slice(
    run_ext,
    x_min=-4000.0,
    x_max=4000.0,
    z_min=-3000.0,
    z_max=5000.0,
    dx=60.0,
    plane='y',
    plane_value=0.0,
    channel='te',
    show_progress=True,
)

nf_tm = pcl.compute_near_field_slice(
    run_ext,
    x_min=-4000.0,
    x_max=4000.0,
    z_min=-3000.0,
    z_max=5000.0,
    dx=60.0,
    plane='y',
    plane_value=0.0,
    channel='tm',
    show_progress=True,
)

a_te, a_tm = run_ext.polarization_jones
nf_mixed = pcl.mix_near_field_slices(nf_te, nf_tm, a_te=a_te, a_tm=a_tm)

# Requested circular-like Jones channel (coherent TE/TM mixing)
fig, _ = pcl.io.plot_nearfield_panels(
    nf_mixed.axis_0,
    nf_mixed.axis_1,
    nf_mixed.field_maps['total'][0],
    nf_mixed.field_maps['total'][1],
    run_ext.positions,
    run_ext.radii,
    plane='y',
    plane_value=0.0,
    real_limits=(-2.0, 2.0),
    abs_limits=(0.0, 2.0),
)
fig.suptitle('Requested Jones near field (coherent TE/TM mixing)', y=1.02)
plt.show()

# TE/TM basis channels
fig, _ = pcl.io.plot_nearfield_panels_channels(
    nf_te.axis_0,
    nf_te.axis_1,
    {
        'TE basis': nf_te.field_maps['total'],
        'TM basis': nf_tm.field_maps['total'],
    },
    run_ext.positions,
    run_ext.radii,
    plane='y',
    plane_value=0.0,
    real_limits=(-2.0, 2.0),
    abs_limits=(0.0, 2.0),
)
plt.show()

# Unpolarized near-field intensity: incoherent average over basis intensities
Iu_nf = pcl.io.unpolarized_near_field_intensity(
    nf_te.field_maps['total'][0],
    nf_tm.field_maps['total'][0],
)
fig, ax = plt.subplots(1, 1, figsize=(10, 5), constrained_layout=True)
pcl.io.plot_intensity(
    ax,
    nf_te.axis_0,
    nf_te.axis_1,
    Iu_nf,
    title=r'Unpolarized near-field intensity $(|E_\text{TE}|^2 + |E_\text{TM}|^2)/2$',
    axis_0_label=nf_te.axis_0_label,
    axis_1_label=nf_te.axis_1_label,
)
pcl.io.plot_spheres(ax, run_ext.positions, run_ext.radii, plane='y', plane_value=0.0, alpha=0.7, color='w')
plt.show()


In [ ]:
# --- Mixed vs unpolarized far-field intensity ---

I_mixed = pcl.io.far_field_intensity_from_result(run_ext, channel='mixed')
I_unpolarized = pcl.io.far_field_intensity_from_result(run_ext, channel='unpolarized')

fig, _ = pcl.io.plot_farfield_hemispheres(
    polar_angles=cfg_ext.polar_angles,
    azimuthal_angles=cfg_ext.azimuthal_angles,
    intensity=I_mixed,
    cmap='magma',
    independent_scales=True,
    title='Mixed (requested Jones) far-field intensity',
)
plt.show()

fig, _ = pcl.io.plot_farfield_hemispheres(
    polar_angles=cfg_ext.polar_angles,
    azimuthal_angles=cfg_ext.azimuthal_angles,
    intensity=I_unpolarized,
    cmap='viridis',
    independent_scales=True,
    title='Unpolarized far-field intensity (0.5 * (TE + TM))',
)
plt.show()


In [ ]:
# --- Save rich output and load it back ---

out_h5 = Path('outputs/01_pyceles_main_replication_with_basis.h5')
pcl.save_simulation_h5(run_ext, nf_te, out_h5)
print('Saved:', out_h5.resolve())

loaded = pcl.io.load_simulation_h5(out_h5)
print('Loaded top-level keys:', list(loaded.keys()))
if 'diagnostics' in loaded:
    print('Diagnostics keys:', list(loaded['diagnostics'].keys()))


### Dipole LDOS / Purcell map on the same near-field slice

This section samples dipole-emitter enhancement on non-internal pixels of the existing slice grid.

- One 3-RHS solve per sampled pixel (`px`, `py`, `pz`) using `solve_sources(...)` + `postprocess_sources(...)`
- pyceles internal verbosity/progress is disabled
- a single `tqdm` tracks sampled pixels
- a pilot solve is executed first to estimate wall-time (ETA) before launching the full map

Note: for large systems (for example 500 particles at `lmax=3`), full-resolution maps can be very expensive.
Use `ldos_stride`, `ldos_max_pixels`, and/or lower `ldos_lmax` to start.

In [ ]:
from time import perf_counter
from tqdm.auto import tqdm

# --- Controls ---
ldos_plane_slice = nf_slice  # reuse the same near-field slice grid
ldos_stride = 1              # >1 for subsampling
ldos_max_pixels = None       # int for quick preview
ldos_run_full_map = False    # False: pilot + ETA only

# Dipole-per-pixel simulation controls
ldos_lmax = max(3, int(run.config.lmax))
ldos_solver_method = 'direct'  # as requested for this showcase
ldos_solver_rtol = float(run.config.solver_rtol)
ldos_moment_magnitude = 1.0 + 0j

inside = np.asarray(ldos_plane_slice.inside, dtype=bool)
valid = ~inside
ii, jj = np.indices(valid.shape)
sampled = valid & (ii % int(ldos_stride) == 0) & (jj % int(ldos_stride) == 0)
pixel_ids = np.argwhere(sampled)
if ldos_max_pixels is not None:
    pixel_ids = pixel_ids[: int(ldos_max_pixels)]

n_valid = int(np.count_nonzero(valid))
n_sampled = int(pixel_ids.shape[0])
print(f'Non-internal pixels on slice: {n_valid}')
print(f'Sampled pixels (stride={ldos_stride}): {n_sampled}')
if n_sampled == 0:
    raise RuntimeError('No sampled non-internal pixels. Reduce stride or check slice/mask settings.')


def _point_xyz(i: int, j: int) -> np.ndarray:
    x = float(ldos_plane_slice.axis_0[i, j])
    z = float(ldos_plane_slice.axis_1[i, j])
    return np.asarray([x, float(ldos_plane_slice.plane_value), z], dtype=float)


# Build one reusable Simulation (operator/precompute can now be reused internally).
probe = pcl.DipoleSource(
    wavelength=float(run.config.wavelength),
    medium_n=complex(run.config.n_medium),
    position=(0.0, float(ldos_plane_slice.plane_value), 0.0),
    dipole_moment=(ldos_moment_magnitude, 0.0 + 0j, 0.0 + 0j),
)
cfg_d = pcl.SimulationConfig(
    wavelength=float(run.config.wavelength),
    n_medium=complex(run.config.n_medium),
    lmax=int(ldos_lmax),
    source=probe,
    solver_method=ldos_solver_method,
    solver_rtol=float(ldos_solver_rtol),
    compute_dtype=run.config.compute_dtype,
    accum_dtype=run.config.accum_dtype,
    verbose=False,
)
sim_d = pcl.Simulation(cfg_d, positions=positions, radii=radii, n_particle=n_particle)


def _solve_pixel_ldos_enhancements(point_xyz: np.ndarray) -> dict[str, float]:
    base = pcl.DipoleSource(
        wavelength=float(run.config.wavelength),
        medium_n=complex(run.config.n_medium),
        position=tuple(point_xyz.tolist()),
        dipole_moment=(ldos_moment_magnitude, 0.0 + 0j, 0.0 + 0j),
    )

    solved = sim_d.solve_sources(
        base.cartesian_basis_sources(
            labels=('px', 'py', 'pz'),
            moment_magnitude=ldos_moment_magnitude,
        ),
        solver_compute_final_residual=False,
    )
    multi = sim_d.postprocess_sources(
        solved,
        include_farfield=False,
    )

    # Public helper: enhancement = P/P0 (projected electric LDOS / Purcell-like ratio).
    return {
        label: float(
            pcl.compute_dipole_ldos_enhancement(
                multi[label],
                channel='mixed',
                allow_inside_particle=False,
                show_progress=False,
            )
        )
        for label in ('px', 'py', 'pz')
    }


# Pilot solve for ETA estimate (always run)
pilot_i, pilot_j = pixel_ids[0]
pilot_xyz = _point_xyz(int(pilot_i), int(pilot_j))
t0 = perf_counter()
pilot = _solve_pixel_ldos_enhancements(pilot_xyz)
pilot_dt = perf_counter() - t0
eta_s = pilot_dt * n_sampled
print('Pilot LDOS/Purcell enhancements at first sampled pixel:', pilot)
print(f'Pilot runtime: {pilot_dt:.2f} s/pixel (3-RHS)')
print(f'Estimated full-map runtime (with solve-only reuse): {eta_s/60:.1f} min ({eta_s/3600:.2f} h)')

# Optional full map
ldos_px = np.full(valid.shape, np.nan, dtype=float)
ldos_py = np.full(valid.shape, np.nan, dtype=float)
ldos_pz = np.full(valid.shape, np.nan, dtype=float)

if ldos_run_full_map:
    for i, j in tqdm(pixel_ids, total=n_sampled, desc='Dipole pixels', unit='px'):
        enh = _solve_pixel_ldos_enhancements(_point_xyz(int(i), int(j)))
        ldos_px[int(i), int(j)] = float(enh['px'])
        ldos_py[int(i), int(j)] = float(enh['py'])
        ldos_pz[int(i), int(j)] = float(enh['pz'])

    ldos_avg = (ldos_px + ldos_py + ldos_pz) / 3.0

    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    panel_data = [
        (ldos_px, 'Projected LDOS/Purcell enhancement (px)'),
        (ldos_py, 'Projected LDOS/Purcell enhancement (py)'),
        (ldos_pz, 'Projected LDOS/Purcell enhancement (pz)'),
        (ldos_avg, 'Orientation-averaged enhancement'),
    ]
    for ax, (data, title) in zip(axes.ravel(), panel_data):
        pcl.io.plot_intensity(
            ax,
            ldos_plane_slice.axis_0,
            ldos_plane_slice.axis_1,
            data,
            title=title,
            axis_0_label=ldos_plane_slice.axis_0_label,
            axis_1_label=ldos_plane_slice.axis_1_label,
        )
        pcl.io.plot_spheres(
            ax,
            run.positions,
            run.radii,
            plane=ldos_plane_slice.plane,
            plane_value=ldos_plane_slice.plane_value,
            alpha=0.7,
            color='w',
        )
    plt.show()
else:
    print('Full map skipped (set ldos_run_full_map=True to run all sampled pixels).')

